# Semantic simialrity
running in a separate notebook to work on two tasks at once. decided to use gemma

In [1]:
import ollama
import json
import numpy as np
import random
import re
import time
from pathlib import Path
from langchain_huggingface import HuggingFaceEmbeddings
import chromadb

PROJECT_ROOT = Path("C:/Users/filip/Desktop/Thesis/project")
DATA_DIR = PROJECT_ROOT / "data"

embed_model = HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 317.35it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
def generate_questions(chunk_text, model="gemma3:1b", num_questions=3):
    """Generate questions that the chunk should be able to answer."""
    prompt = (
        
        f"Based on the following scientific text, generate exactly {num_questions} "
        "questions that can be answered using ONLY this text. "
        "Return ONLY a JSON list of strings. No explanation, no markdown.\n\n"
        f"Text: {chunk_text}"
    )
    response = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0.7},
    )
    raw = response["message"]["content"].strip()
    if raw.startswith("```"):
        raw = raw.split("\n", 1)[1] if "\n" in raw else raw[3:]
        if "```" in raw:
            raw = raw.rsplit("```", 1)[0]
    raw = raw.strip()
    
    questions = json.loads(raw)
    if not isinstance(questions, list):
        raise ValueError(f"Expected list, got {type(questions)}")
    return [str(q) for q in questions]

In [5]:
test_path = DATA_DIR / "chunks" / "recursive_1000" / "algae-2020-35-5-31.json"
with open(test_path, encoding="utf-8") as f:
    doc = json.load(f)

chunk = doc["chunks"][0]
questions = generate_questions(chunk["text"])
print("Questions:")
for q in questions:
    print(f"  {q}")

Questions:
  What is the new species name described in this paper?
  What is the family of the algae?
  What is the phylum of the algae?


results for gemma 1b:
  What is the name of the species being investigated?
  Where was the species discovered?
  What is the primary focus of the study?